# Date Parsing and Datetime Basics
Task 21 - Day 21

In [1]:
import pandas as pd

# dates in several different formats, plus one invalid entry
df = pd.DataFrame({
    "OrderID": [101, 102, 103, 104, 105, 106, 107],
    "OrderDate": ["2023-01-15", "2023/02/20", "15-03-2023", "2023-04-10",
                   "not_a_date", "2023-06-05", "2023-07-22"],
    "Amount": [1200, 1500, 900, 2100, 1750, 1300, 1600],
})
df

,OrderID,OrderDate,Amount
0,101,2023-01-15,1200
1,102,2023/02/20,1500
2,103,15-03-2023,900
3,104,2023-04-10,2100
4,105,not_a_date,1750
5,106,2023-06-05,1300
6,107,2023-07-22,1600


## Date conversion examples

In [2]:
print(df["OrderDate"].dtype)

str


In [3]:
# format="mixed" lets Pandas infer each row's format individually,
# errors="coerce" turns anything unparseable into NaT instead of crashing
df["OrderDate_clean"] = pd.to_datetime(df["OrderDate"], errors="coerce", format="mixed")
df

,OrderID,OrderDate,Amount,OrderDate_clean
0,101,2023-01-15,1200,2023-01-15
1,102,2023/02/20,1500,2023-02-20
2,103,15-03-2023,900,2023-03-15
3,104,2023-04-10,2100,2023-04-10
4,105,not_a_date,1750,NaT
5,106,2023-06-05,1300,2023-06-05
6,107,2023-07-22,1600,2023-07-22


In [4]:
print(df.dtypes)

OrderID                     int64
OrderDate                     str
Amount                      int64
OrderDate_clean    datetime64[us]
dtype: object


In [5]:
# checking for values that failed to parse
print(df[df["OrderDate_clean"].isna()])

   OrderID   OrderDate  Amount OrderDate_clean
4      105  not_a_date    1750             NaT


In [6]:
# parsing a column that is entirely one known format
specific_format = pd.Series(["15-03-2023", "20-04-2023"])
parsed = pd.to_datetime(specific_format, format="%d-%m-%Y")
print(parsed)

0   2023-03-15
1   2023-04-20
dtype: datetime64[us]


## Extracting year, month, day, and weekday

In [7]:
df["Year"] = df["OrderDate_clean"].dt.year
df["Month"] = df["OrderDate_clean"].dt.month
df["Day"] = df["OrderDate_clean"].dt.day
df["Weekday"] = df["OrderDate_clean"].dt.day_name()
df

,OrderID,OrderDate,Amount,OrderDate_clean,Year,Month,Day,Weekday
0,101,2023-01-15,1200,2023-01-15,2023.0,1.0,15.0,Sunday
1,102,2023/02/20,1500,2023-02-20,2023.0,2.0,20.0,Monday
2,103,15-03-2023,900,2023-03-15,2023.0,3.0,15.0,Wednesday
3,104,2023-04-10,2100,2023-04-10,2023.0,4.0,10.0,Monday
4,105,not_a_date,1750,NaT,NaN,NaN,NaN,NaN
5,106,2023-06-05,1300,2023-06-05,2023.0,6.0,5.0,Monday
6,107,2023-07-22,1600,2023-07-22,2023.0,7.0,22.0,Saturday


## A short analysis based on dates

In [8]:
# how many days ago each order was placed
today = pd.Timestamp("2026-10-07")
df["DaysAgo"] = (today - df["OrderDate_clean"]).dt.days
print(df[["OrderID", "OrderDate_clean", "DaysAgo"]])

   OrderID OrderDate_clean  DaysAgo
0      101      2023-01-15   1361.0
1      102      2023-02-20   1325.0
2      103      2023-03-15   1302.0
3      104      2023-04-10   1276.0
4      105             NaT      NaN
5      106      2023-06-05   1220.0
6      107      2023-07-22   1173.0


In [9]:
# filtering by a date range
mask = (df["OrderDate_clean"] >= "2023-03-01") & (df["OrderDate_clean"] <= "2023-06-01")
print(df[mask])

   OrderID   OrderDate  Amount OrderDate_clean    Year  Month   Day  \
2      103  15-03-2023     900      2023-03-15  2023.0    3.0  15.0   
3      104  2023-04-10    2100      2023-04-10  2023.0    4.0  10.0   

     Weekday  DaysAgo  
2  Wednesday   1302.0  
3     Monday   1276.0  


In [10]:
# total order amount per month
monthly_totals = df.dropna(subset=["OrderDate_clean"]).groupby("Month")["Amount"].sum()
print(monthly_totals)

Month
1.0    1200
2.0    1500
3.0     900
4.0    2100
6.0    1300
7.0    1600
Name: Amount, dtype: int64


Out of 7 orders, only order 105 had an unparseable date and was converted to a missing value instead of crashing the whole conversion. Orders cluster across six different months with no obvious seasonal pattern in the small sample here, and April (order 104) had the single largest order value at 2100.

## Interview questions

**How do you convert a column to datetime?**
With `pd.to_datetime(column)`. If the dates are in a single known format, passing `format="%d-%m-%Y"` (or whatever matches) is faster and stricter. If formats are mixed or might contain bad values, `format="mixed"` lets Pandas infer each row individually, and `errors="coerce"` turns anything unparseable into `NaT` (Not a Time) instead of raising an error.

**What is the .dt accessor?**
A namespace on a datetime Series that exposes date and time components and methods — `.dt.year`, `.dt.month`, `.dt.day`, `.dt.day_name()`, and more. It only works on a column that's actually been converted to a datetime dtype; calling `.dt` on a column still stored as text raises an error.

**Why is date preprocessing important?**
Dates stored as plain text sort alphabetically, not chronologically, and can't have components like month or weekday extracted. Converting to datetime enables correct sorting, filtering by date range, time-based grouping (like the monthly totals above), and calculating durations between dates — none of which work reliably on raw strings, especially when the source data mixes formats or contains invalid entries.